In [ ]:
# CELL 1 - Check GPU
import tensorflow as tf
gpus = tf.config.list_physical_devices('GPU')
print('GPU ON:', gpus) if gpus else print('NO GPU - Go to Runtime > Change runtime type > T4 GPU > Save > Restart')

In [ ]:
# CELL 2 - Mount Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# CELL 3 - Download mulberry dataset
!pip install kaggle -q
from google.colab import files
print('Upload kaggle.json:')
files.upload()
!mkdir -p ~/.kaggle && cp kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json
!kaggle datasets download -d nahiduzzaman13/mulberry-leaf-dataset -p /content/mulberry --unzip
print('Done!')

In [ ]:
# CELL 4 - Show dataset structure
import os
for root, dirs, files in os.walk('/content/mulberry'):
    level = root.replace('/content/mulberry', '').count(os.sep)
    print('  ' * level + os.path.basename(root) + '/ (' + str(len(files)) + ' files)')
    if level >= 3:
        break

In [ ]:
# CELL 5 - Find dataset root
import os
dataset_path = None
for root, dirs, files_list in os.walk('/content/mulberry'):
    valid = []
    for d in dirs:
        sub = os.path.join(root, d)
        imgs = [f for f in os.listdir(sub) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
        if len(imgs) > 5:
            valid.append((d, len(imgs)))
    if len(valid) >= 2:
        dataset_path = root
        print('Dataset root:', root)
        for name, count in valid:
            print('  ' + name + ': ' + str(count) + ' images')
        break

In [ ]:
# CELL 6 - Rename to clean class names
import os, shutil
rename_map = {
    'Disease Free leaves': 'Healthy',
    'Disease Free Leaves': 'Healthy',
    'disease free leaves': 'Healthy',
    'Leaf Rust': 'Leaf Rust',
    'leaf rust': 'Leaf Rust',
    'Leaf spot': 'Leaf Spot',
    'Leaf Spot': 'Leaf Spot',
    'leaf spot': 'Leaf Spot',
}
CLEAN_DIR = '/content/mulberry_clean'
if os.path.exists(CLEAN_DIR):
    shutil.rmtree(CLEAN_DIR)
os.makedirs(CLEAN_DIR)
for folder in os.listdir(dataset_path):
    src = os.path.join(dataset_path, folder)
    if not os.path.isdir(src):
        continue
    clean_name = rename_map.get(folder, folder)
    dst = os.path.join(CLEAN_DIR, clean_name)
    shutil.copytree(src, dst)
    print('"' + folder + '" -> "' + clean_name + '" (' + str(len(os.listdir(dst))) + ' images)')
print('\nFinal classes:')
for cls in sorted(os.listdir(CLEAN_DIR)):
    print('  ' + cls + ': ' + str(len(os.listdir(os.path.join(CLEAN_DIR, cls)))) + ' images')

In [ ]:
# CELL 7 - Verify images load correctly
import tensorflow as tf
import matplotlib.pyplot as plt
import numpy as np
check_ds = tf.keras.utils.image_dataset_from_directory(
    CLEAN_DIR, image_size=(224, 224), batch_size=9, shuffle=True
)
class_names = check_ds.class_names
print('Classes in order:', class_names)
plt.figure(figsize=(10, 10))
for images, labels in check_ds.take(1):
    for i in range(9):
        plt.subplot(3, 3, i + 1)
        plt.imshow(images[i].numpy().astype('uint8'))
        plt.title(class_names[np.argmax(labels[i])])
        plt.axis('off')
plt.tight_layout()
plt.show()
print('If images match their labels above, proceed to CELL 8')

In [ ]:
# CELL 8 - Train Phase 1
from tensorflow import keras
from keras import layers, models
from keras.applications import MobileNetV2
import tensorflow as tf

IMG_SIZE = (224, 224)
BATCH_SIZE = 8
EPOCHS = 50
MODEL_PATH = '/content/drive/MyDrive/mulberry_model.h5'

augment = keras.Sequential([
    layers.RandomFlip('horizontal_and_vertical'),
    layers.RandomRotation(0.4),
    layers.RandomZoom(0.3),
    layers.RandomBrightness(0.3),
    layers.RandomContrast(0.3),
    layers.RandomTranslation(0.1, 0.1),
])

train_ds = tf.keras.utils.image_dataset_from_directory(
    CLEAN_DIR, validation_split=0.2, subset='training',
    seed=42, image_size=IMG_SIZE, batch_size=BATCH_SIZE, label_mode='categorical'
)
val_ds = tf.keras.utils.image_dataset_from_directory(
    CLEAN_DIR, validation_split=0.2, subset='validation',
    seed=42, image_size=IMG_SIZE, batch_size=BATCH_SIZE, label_mode='categorical'
)

class_names = train_ds.class_names
num_classes = len(class_names)
print('Training classes:', class_names)

with open('/content/drive/MyDrive/class_names.txt', 'w') as f:
    f.write('\n'.join(class_names))
print('class_names.txt saved!')

total = 440 + 489 + 162
class_weight = {
    class_names.index('Healthy'): total / (3 * 440),
    class_names.index('Leaf Rust'): total / (3 * 489),
    class_names.index('Leaf Spot'): total / (3 * 162),
}
print('Class weights:', class_weight)

train_ds = train_ds.map(
    lambda x, y: (augment(x / 255.0, training=True), y),
    num_parallel_calls=tf.data.AUTOTUNE
).prefetch(tf.data.AUTOTUNE)

val_ds = val_ds.map(
    lambda x, y: (x / 255.0, y),
    num_parallel_calls=tf.data.AUTOTUNE
).prefetch(tf.data.AUTOTUNE)

base = MobileNetV2(input_shape=(224, 224, 3), include_top=False, weights='imagenet')
base.trainable = False

model = models.Sequential([
    base,
    layers.GlobalAveragePooling2D(),
    layers.BatchNormalization(),
    layers.Dense(256, activation='relu'),
    layers.Dropout(0.5),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.4),
    layers.Dense(num_classes, activation='softmax')
])

model.compile(
    optimizer=keras.optimizers.Adam(1e-3),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

callbacks = [
    keras.callbacks.EarlyStopping(patience=8, restore_best_weights=True, verbose=1),
    keras.callbacks.ReduceLROnPlateau(factor=0.3, patience=4, verbose=1),
    keras.callbacks.ModelCheckpoint(MODEL_PATH, save_best_only=True, verbose=1)
]

print('\nPhase 1: Training top layers...')
model.fit(
    train_ds, validation_data=val_ds,
    epochs=EPOCHS, callbacks=callbacks,
    class_weight=class_weight
)

In [ ]:
# CELL 9 - Fine-tune Phase 2
print('Phase 2: Fine-tuning top 50 layers...')
base.trainable = True
for layer in base.layers[:-50]:
    layer.trainable = False

model.compile(
    optimizer=keras.optimizers.Adam(1e-5),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

model.fit(
    train_ds, validation_data=val_ds,
    epochs=30, callbacks=callbacks,
    class_weight=class_weight
)

loss, acc = model.evaluate(val_ds, verbose=0)
print('\nFinal Accuracy: ' + str(round(acc * 100, 2)) + '%')
print('Classes:', class_names)
print('Model saved to Google Drive!')

In [ ]:
# CELL 10 - Test predictions visually
import numpy as np
import matplotlib.pyplot as plt

loaded_model = tf.keras.models.load_model(MODEL_PATH)
plt.figure(figsize=(12, 8))
count = 0
for images, labels in val_ds.take(3):
    preds = loaded_model.predict(images, verbose=0)
    for i in range(min(3, len(images))):
        count += 1
        if count > 9:
            break
        plt.subplot(3, 3, count)
        img = (images[i].numpy() * 255).astype('uint8')
        plt.imshow(img)
        true_label = class_names[np.argmax(labels[i])]
        pred_label = class_names[np.argmax(preds[i])]
        conf = round(float(np.max(preds[i])) * 100, 1)
        color = 'green' if true_label == pred_label else 'red'
        plt.title('True: ' + true_label + '\nPred: ' + pred_label + ' (' + str(conf) + '%)', color=color, fontsize=8)
        plt.axis('off')
plt.tight_layout()
plt.show()
print('Green = correct prediction, Red = wrong prediction')